# Dealing with Categorical Variables

Many real datasets describe things with categories rather than numbers: a car's country of origin, a shop's location, a house's condition. A regression model needs numbers, so this notebook shows how to turn categories into features it can use, and how to read the coefficients that come out.

## Learning Objectives

At the end of this notebook, you should be able to:

- Identify the categorical variables in a dataset, including ones stored as numbers.
- Encode a categorical variable as dummy variables, dropping one column to avoid the dummy variable trap.
- Calculate the difference between a category and the reference category from a dummy coefficient.
- Compare a model with and without dummy variables using adjusted $R^2$.

To practise all this, we stick with the cars data we used for multiple linear regression.

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Import data
data = pd.read_csv("data/cars_multivariate.csv")
data.head()

In [ ]:
# Check data type for each column
data.info()

Except for `car_name`, every column holds numerical values, so each one could be a candidate feature for predicting the dependent variable `mpg` (miles per gallon). At first glance, there seem to be no categorical variables in this dataset.

## What are categorical variables?


Usually we know the nature of the data, and therefore which variables are categorical and which are numerical. Even when we do not, we can explore the data to find out. Let's take a closer look at the `origin` column.

In [ ]:
# Descriptive statistics for column origin
print(data["origin"].describe())

In [ ]:
# Number of unique values in column origin
print(data["origin"].nunique())

In [ ]:
# Unique values in column origin
print(data["origin"].unique())

The values range from 1 to 3, and in fact the only values in the dataset are 1, 2 and 3. This shows that `origin` is a **categorical variable**. It does not represent a continuous quantity but refers to a location: 1 may stand for US, 2 for Europe and 3 for Asia (the actual meaning is not disclosed for this dataset).

Categorical variables are exactly what they sound like: they represent categories, not numerical quantities. Although it is not the case here, categorical features are often stored as text values that name the different levels.

## Identifying categorical variables

Categorical variables need special treatment, as you will see later on, so you first need to identify which variables are categorical.

In some cases this is easy, for example when they are stored as strings. In other cases they are stored as numbers, and the fact that they are categorical is not immediately apparent.

Start with the `.info()` and `.describe()` methods. `.info()` shows the data type of each column (strings, integers and so on), and `.describe()` shows the descriptive statistics. Even so, continuous variables are sometimes imported as strings, so always look at the data itself, for example with plots like the ones below.

In [ ]:
# Create plots to identify categorical features
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 3))

for xcol, ax in zip(["acceleration", "displacement", "horsepower", "weight"], axes):
    data.plot(kind="scatter", x=xcol, y="mpg", ax=ax, alpha=0.4, color="b")

In [ ]:
# Create plots to identify categorical features
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 3))

for xcol, ax in zip(["cylinders", "model_year", "origin"], axes):
    data.plot(kind="scatter", x=xcol, y="mpg", ax=ax, alpha=0.4, color="b")

Compare the structure of the first and the second set of plots. The continuous features in the first set produce a fairly homogeneous "cloud" of points, while the categorical features in the second set produce vertical lines at a few discrete values.

A histogram is another useful plot for spotting categorical variables:

In [ ]:
data.hist(figsize=(8, 8));

Finally, count the number of unique values in these three columns:

In [ ]:
data[["cylinders", "model_year", "origin"]].nunique()

## Dummy Variables

A regression model multiplies each feature by a coefficient, so it treats every number as a quantity. Coded as 1, 2 and 3, `origin` would tell the model that origin 3 is "three times" origin 1, which makes no sense for countries.

Instead, we turn each category into its own column that is 1 when the row belongs to that category and 0 otherwise. These columns are called **dummy variables**, and the technique is also called **one-hot encoding**.

> [!NOTE]
> Another technique, **label encoding**, replaces each category with an integer. It suits tree-based models, not linear regression. You will meet it, together with scikit-learn's encoders, in W3D3 (Feature Engineering).

To illustrate dummy variables, start with a small pandas Series with 3 categories: `"USA"`, `"EU"` and `"ASIA"`.

In [ ]:
origin = ["USA", "EU", "EU", "ASIA", "USA", "EU", "EU", "ASIA", "ASIA", "USA"]
origin_series = pd.Series(origin)

First, tell pandas that these strings are categories:

In [ ]:
cat_origin = origin_series.astype("category")
cat_origin

Note that the `dtype` (data type) is now `category` and that pandas has detected the three categories.

pandas creates the dummy variables with `pd.get_dummies()`:

Documentation: [`get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html).

In [ ]:
# Create dummy variables using pandas .get_dummies()
pd.get_dummies(cat_origin)

Each category has become its own column. Because dummy variables are only ever 0 or 1, the model cannot misread them as a continuous quantity. For linear regression, and for most other models in scikit-learn, categorical features must be encoded like this before you can use them.

## The Dummy Variable Trap



Because of the way dummy variables are created, any one of them can be predicted from all the others. This is known as perfect **multicollinearity**, and it is a problem for regression.

Perfect multicollinearity means that you can *perfectly* predict one variable from a combination of the other variables. To see this, go back to the one-hot encoded origin data from above:

In [ ]:
# Create dummy variables using pandas .get_dummies()
trap_df = pd.get_dummies(cat_origin)
trap_df.head()

Because there is a dummy variable for every origin, you can predict any one of them from the others. Take the `ASIA` column: add the values in the `EU` and `USA` columns and subtract the sum from 1. The result is exactly the `ASIA` column:

In [ ]:
# Predict ASIA column from EU and USA
predicted_asia = 1 - (trap_df["EU"] + trap_df["USA"])
predicted_asia.to_frame(name="Predicted_ASIA")

You can predict `EU` and `USA` in the same way; try it on your own.

Why is this a problem for regression? In a multiple linear regression, each coefficient represents the average change in the dependent variable for a 1-unit change in that predictor, **assuming all other predictors are held constant**. When the predictors are perfectly related, as dummy variables are, one of them cannot change while the others stay constant, so the coefficients can no longer be estimated or interpreted in this way. This is known as the **dummy variable trap**.

You avoid the dummy variable trap by dropping one of the dummy variables. You can drop a column from the DataFrame manually or, more conveniently, pass `drop_first=True` to `get_dummies()`:

In [ ]:
pd.get_dummies(cat_origin, drop_first=True)

In the DataFrame above, there is no longer enough information to predict any one column from the others, so the multicollinearity is gone.

Dropping the first variable affects how you read the regression coefficients. The dropped category becomes the **reference category**, and the coefficients of the remaining dummy variables represent the change *relative* to that reference. You will see this later in the notebook.

In some contexts, multicollinearity and the dummy variable trap matter less and can be ignored. It is therefore important to know which models are sensitive to multicollinearity and which are not.

## Back to our cars data

Now convert the `cylinders`, `model_year` and `origin` columns into dummy variables, dropping the first category of each.

In [ ]:
# Create dummy variables for categorical columns
cyl_dummies = pd.get_dummies(data["cylinders"], prefix="cyl", drop_first=True)
yr_dummies = pd.get_dummies(data["model_year"], prefix="yr", drop_first=True)
orig_dummies = pd.get_dummies(data["origin"], prefix="orig", drop_first=True)

Next, remove the original columns from the data and add the dummy columns instead:

In [ ]:
# Drop original columns
data = data.drop(["cylinders", "model_year", "origin"], axis=1)

In [ ]:
# Concatenate dummy variables to dataframe
data = pd.concat([data, cyl_dummies, yr_dummies, orig_dummies], axis=1)
data.head()

## Does the Encoding Help?

Now train two models and compare them: one on the numeric features only, as in notebook 03, and one that also uses the dummy columns. Adjusted $R^2$ tells us whether the extra columns add real explanatory power.

Documentation: [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [ ]:
from sklearn.linear_model import LinearRegression


def adjusted_r_squared(r_squared, X):
    return 1 - (1 - r_squared) * (len(X) - 1) / (len(X) - X.shape[1] - 1)


y = data["mpg"]
X_numeric = data[["displacement", "horsepower", "weight", "acceleration"]]
X_dummies = data.drop(columns=["mpg", "car_name"])

for name, X in [("numeric only", X_numeric), ("numeric + dummies", X_dummies)]:
    r2 = LinearRegression().fit(X, y).score(X, y)
    print(f"{name:18} features: {X.shape[1]:2}  R2: {r2:.3f}  adjusted R2: {adjusted_r_squared(r2, X):.3f}")

Adding the 18 dummy columns raises $R^2$ from 0.707 to 0.874, and adjusted $R^2$ from 0.704 to 0.867. Because adjusted $R^2$ rises as well, the gain is not just the effect of adding more columns: the model year, the number of cylinders and the origin explain a large part of the fuel consumption that the numeric features miss.

## Reading a Dummy Coefficient

Each dummy coefficient is a difference from the **reference category**, the one we dropped. Here the reference is origin 1. Start with a model that uses only the origin dummies:

In [ ]:
X_origin = orig_dummies
origin_only = LinearRegression().fit(X_origin, y)

print(f"intercept: {origin_only.intercept_:.2f}")
print(pd.Series(origin_only.coef_, index=X_origin.columns).round(2))

# Compare with the average mpg of each origin, from the original data
original = pd.read_csv("data/cars_multivariate.csv")
original.groupby("origin")["mpg"].mean().round(2)

The intercept, 20.03, is the average `mpg` of origin 1 cars. The coefficients say that origin 2 cars do 7.57 mpg more on average, and origin 3 cars 10.42 mpg more, than origin 1 cars. With only dummies in the model, these are exactly the differences between the group averages.

Now add `weight` to the model:

In [ ]:
X_weight_origin = pd.concat([data[["weight"]], orig_dummies], axis=1)
weight_origin = LinearRegression().fit(X_weight_origin, y)

pd.Series(weight_origin.coef_, index=X_weight_origin.columns).round(3)

The origin coefficients shrink to 0.971 and 2.327. Origin 2 and 3 cars are much lighter on average, and much of their better fuel consumption comes from that. Once `weight` is in the model, each coefficient compares cars **of the same weight**: an origin 3 car does about 2.3 mpg more than an origin 1 car that weighs the same. This is the "holding the other features fixed" reading from notebook 03, applied to categories.

## Try It

Go back to exercise 04 (car seats) and add `ShelveLoc`, `Urban` and `US` as dummy variables, dropping the first category of each. Does the adjusted $R^2$ of your model improve?

## Summary

In this notebook you:

- Identified the **categorical variables** in a dataset by exploring it: the `info()` output, descriptive statistics, the number of unique values, and scatter plots and histograms.
- Encoded categorical variables as **dummy variables** with `pd.get_dummies()`, and avoided the **dummy variable trap** by dropping one column (`drop_first=True`). A full set of dummy columns is perfectly collinear, because any one column can be predicted from the others.
- Calculated the difference between each category and the **reference category** from the dummy coefficients, first alone and then holding `weight` fixed.
- Compared the models with and without dummies: adjusted $R^2$ rose from 0.704 to 0.867.